  <a href="https://colab.research.google.com/github/marcpalo1999/MIA_sanidad/blob/main/2_3_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   # Aprendizaje Supervisado para Clasificación Médica: Predicción de Enfermedad Coronaria



   ## Objetivos



   En este script construiremos modelos de Machine Learning para clasificación médica usando el dataset de enfermedades cardíacas (nivel de estenosis coronaria) para predecir qué pacientes necesitan cateterismo cardíaco y cuales se lo pueden ahorrar.

In [ ]:
import os

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    if not os.path.exists('/content/MIA_sanidad'):
        !git clone https://github.com/marcpalo1999/MIA_sanidad.git /content/MIA_sanidad
    os.chdir('/content/MIA_sanidad')

print(os.getcwd())

In [ ]:
# Importar librerías necesarias dandoles un alias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (confusion_matrix, accuracy_score, roc_auc_score,
                             precision_score, recall_score, f1_score, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)

# Modelos de clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import joblib

   ---



   # SECCIÓN 0: Cargar los datos y dividir entre diseño y validación

In [ ]:
# %%
# Cargar el dataset
df = pd.read_csv("./data/heart_disease_dataset_con_nulos.csv")

print(f"Dataset cargado: {df.shape[0]} pacientes, {df.shape[1]} variables")
df.head()

   # SECCIÓN 0.1: Resumen Descripción del Problema Clínico



   Para un entendimiento completo del problema, leed el archivo markdown adjunto titulado `2_2_ML_Explicación_Datos.md`.



   Contexto del problema:



   La secuencia diagnóstica actual para enfermedad coronaria es:



   1. Evaluación clínica (edad, síntomas, factores de riesgo)



   2. ECG en reposo



   3. Prueba de esfuerzo con ECG



        - Si es ambigua → Gammagrafía de perfusión miocardiaca (SPECT, ~450euros), para descartar



           - si la gammagrafia es positiva vamos a cateterismo terapeutico



           - si es gammagrafia es negativa dejamos en seguimiento/profilaxis



        - Si positiva → Cateterismo, primeramente diagnostico (INVASIVO, riesgos, coste alto ~3000€)



   Problema clínico:



   - Muchos pacientes se someten a cateterismo innecesariamente



   - El cateterismo tiene riesgos (2% complicaciones: hematomas, nefropatía, etc.)



   - Alto coste económico



   Solución propuesta con ML:



   Predecir el resultado del cateterismo usando solo las pruebas no invasivas previas,



   para reducir cateterismos innecesarios manteniendo alta sensibilidad (no perder casos reales de enfermedad).



   Dos estrategias de modelado:



   1. Modelo Básico: Solo consulta + ECG + prueba esfuerzo (aplicable a TODOS los pacientes)



      - Variables: edad, sexo, tipo dolor torácico, presión arterial, colesterol, ECG reposo, resultados prueba esfuerzo



      - Objetivo: Screening inicial para evitar cateterismos innecesarios



   2. Modelo Completo: Incluye gammagrafía (solo para casos equívocos)



      - Variables: todas las anteriores + resultado gammagrafía (thal)



      - Objetivo: Optimizar decisión cuando ya se realizó gammagrafía



   **Data Leakage**



   Data leakage ocurre cuando, durante el diseño del modelo, usamos información que

   en la realidad no estaría disponible en el momento de la predicción. Esto hace que

   el modelo parezca mejor de lo que realmente es — al ponerlo en producción, fallará.

   Hay dos tipos principales: variables que no debemos incluir (como veremos con `ca`)

   y pasos de preprocesamiento realizados con datos del test (imputación, escalado...) 
   
   que hacen que sobreestimemos lo bien que lo hará nuestro modelo en la realidad, ya 
   
   que le estamos dando pistas de comos erá el test durante el proceso de entrenamiento.



   Variable excluida por data leakage:



   - `ca` (número de vasos obstruidos visualizados en fluoroscopia) se obtiene DURANTE el cateterismo



   - Usar `ca` para predecir `num` (resultado cateterismo) es circular, ya que ambas se miden simultáneamente



   - Incluirla sería como hacer trampa: predecir el resultado usando información del mismo procedimiento



   Binarización del target:



   - Variable original `num`: 0 (sin obstrucción coronatia), 1-4 (diferentes grados de obstruccion en diferentes cavidades)



   - Simplificamos a binario: 0 = no necesita cateterismo, 1 = necesita cateterismo



   - Razón clínica: la decisión es binaria (hacer/no hacer cateterismo), no gradual

   ---



   # SECCIÓN 1: Entendiendo el Problema desde ML



   Tipo de problema:



   - Supervised Learning: tenemos variable objetivo (target) para entrenar



   - Classification: la variable objetivo es categórica (enfermedad: sí/no)



   Objetivo: Dado un nuevo paciente con sus características clínicas, predecir si tiene



   enfermedad coronaria significativa que requiera cateterismo.

In [ ]:
from IPython.display import HTML

pipeline_html = """
<style>
  .pipe-wrap {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    font-size: 13px;
    max-width: 720px;
    margin: 24px auto;
    line-height: 1.5;
  }
  .pipe-wrap .title {
    font-size: 15px;
    font-weight: 600;
    margin-bottom: 20px;
    color: #1a1a1a;
  }
  .pipe-row {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 0;
  }
  .pipe-box {
    border-radius: 8px;
    padding: 8px 20px;
    text-align: center;
    width: 340px;
    box-sizing: border-box;
  }
  .pipe-box .label { font-weight: 600; font-size: 13px; }
  .pipe-box .sub   { font-size: 11px; margin-top: 2px; opacity: 0.8; }

  .box-gray   { background: #f0efea; border: 1px solid #c8c7bf; color: #2c2c2a; }
  .box-blue   { background: #e6f1fb; border: 1px solid #85b7eb; color: #0c447c; }
  .box-purple { background: #eeedfe; border: 1px solid #afa9ec; color: #3c3489; }
  .box-coral  { background: #faece7; border: 1.5px solid #d85a30; color: #4a1b0c; font-weight: 700; }
  .box-teal   { background: #e1f5ee; border: 1px solid #5dcaa5; color: #085041; }

  .arrow {
    width: 1px; height: 24px;
    background: #bbb;
    position: relative;
    margin: 0 auto;
  }
  .arrow::after {
    content: '';
    position: absolute;
    bottom: -5px; left: -4px;
    border-left: 5px solid transparent;
    border-right: 5px solid transparent;
    border-top: 6px solid #bbb;
  }

  .split-wrap {
    display: flex;
    justify-content: center;
    align-items: flex-start;
    gap: 0;
    width: 100%;
    margin-top: 0;
  }
  .split-col {
    display: flex;
    flex-direction: column;
    align-items: center;
    width: 300px;
  }
  .split-label {
    font-size: 11px;
    font-weight: 600;
    color: #888;
    margin: 6px 0 4px;
    letter-spacing: 0.04em;
    text-transform: uppercase;
  }
  .pipe-box.narrow { width: 260px; }

  .split-connector {
    display: flex;
    justify-content: center;
    width: 100%;
    position: relative;
    height: 40px;
  }
  .split-connector svg { overflow: visible; }

  .merge-connector {
    display: flex;
    justify-content: center;
    width: 100%;
    height: 40px;
  }

  .dashed-line {
    border-left: 1.5px dashed #bbb;
    height: 100%;
    margin: 0 auto;
  }

  .rule {
    margin: 18px 0 4px;
    padding: 10px 16px;
    background: #fffbf0;
    border-left: 3px solid #ef9f27;
    border-radius: 0 6px 6px 0;
    font-size: 12px;
    color: #3d3d3a;
    max-width: 680px;
  }
  .rule b { color: #854f0b; }
</style>

<div class="pipe-wrap">
  <div class="title">Mapa del pipeline — visión global</div>

  <div class="pipe-row">

    <!-- Datos brutos -->
    <div class="pipe-box box-gray">
      <div class="label">Datos brutos</div>
    </div>
    <div class="arrow"></div>

    <!-- S2 EDA -->
    <div class="pipe-box box-blue">
      <div class="label">§2 — EDA</div>
      <div class="sub">Distribuciones, missings, outliers</div>
    </div>
    <div class="arrow"></div>

    <!-- S3 -->
    <div class="pipe-box box-blue">
      <div class="label">§3 — Variables categóricas</div>
      <div class="sub">Identificar encoding pendiente</div>
    </div>
    <div class="arrow"></div>

    <!-- S4 -->
    <div class="pipe-box box-blue">
      <div class="label">§4 — Estrategias de modelado</div>
    </div>
    <div class="arrow"></div>

    <!-- S5 SPLIT -->
    <div class="pipe-box box-coral" style="width:420px">
      <div class="label">§5 — Train / Test split &nbsp;←&nbsp; punto de corte crítico</div>
    </div>

    <!-- Split connector -->
    <div class="split-connector" style="width:420px">
      <svg width="420" height="40">
        <line x1="210" y1="0" x2="80"  y2="40" stroke="#bbb" stroke-width="1.5" marker-end="url(#arr)"/>
        <line x1="210" y1="0" x2="340" y2="40" stroke="#bbb" stroke-width="1.5" marker-end="url(#arr)"/>
        <defs>
          <marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse">
            <path d="M2 1L8 5L2 9" fill="none" stroke="#bbb" stroke-width="1.5" stroke-linecap="round"/>
          </marker>
        </defs>
      </svg>
    </div>

    <!-- Two columns -->
    <div class="split-wrap" style="width:420px; align-items:flex-start">

      <!-- LEFT: train column -->
      <div class="split-col">
        <div class="split-label">X_train</div>

        <div class="pipe-box box-purple narrow">
          <div class="label">§6.1 — One-Hot Encoding</div>
          <div class="sub">Separado train/test, reindex</div>
        </div>
        <div class="arrow"></div>

        <div class="pipe-box box-purple narrow">
          <div class="label">§6.2 — Imputación</div>
          <div class="sub">Mediana aprendida solo de train</div>
        </div>
        <div class="arrow"></div>

        <div class="pipe-box box-purple narrow">
          <div class="label">§6.3 — Escalado</div>
          <div class="sub">Media/std solo de train</div>
        </div>
        <div class="arrow"></div>

        <div class="pipe-box box-purple narrow">
          <div class="label">§§8–12 — Entrenamiento</div>
          <div class="sub">Baseline + modelos (solo train)</div>
        </div>
      </div>

      <!-- RIGHT: test column (dashed, waits) -->
      <div class="split-col">
        <div class="split-label">X_test</div>
        <div style="height: 220px; display:flex; align-items:center; justify-content:center;">
          <div style="display:flex; flex-direction:column; align-items:center; gap:6px; color:#aaa; font-size:11px;">
            <div class="dashed-line" style="height:180px"></div>
            <span>en espera</span>
          </div>
        </div>
      </div>

    </div><!-- end split-wrap -->

    <!-- Merge connector -->
    <div class="merge-connector" style="width:420px">
      <svg width="420" height="40">
        <line x1="80"  y1="0" x2="210" y2="40" stroke="#bbb" stroke-width="1.5" marker-end="url(#arr2)"/>
        <line x1="340" y1="0" x2="210" y2="40" stroke="#bbb" stroke-width="1.5" marker-end="url(#arr2)"/>
        <defs>
          <marker id="arr2" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse">
            <path d="M2 1L8 5L2 9" fill="none" stroke="#bbb" stroke-width="1.5" stroke-linecap="round"/>
          </marker>
        </defs>
      </svg>
    </div>

    <!-- S13 EVALUACIÓN FINAL -->
    <div class="pipe-box box-coral" style="width:420px">
      <div class="label">§13 — Evaluación final &nbsp;←&nbsp; test usado 1 sola vez</div>
    </div>
    <div class="arrow"></div>

    <!-- S14-16 -->
    <div class="pipe-box box-teal" style="width:420px">
      <div class="label">§§14–16 — ROC · Feature importance · Threshold → Producción</div>
    </div>

  </div><!-- end pipe-row -->

  <div class="rule">
    <b>Regla de oro:</b> Todo lo que "aprende" del dataset (medianas, medias, stds, thresholds)
    se aprende <b>únicamente de los datos de train</b>. El test set es siempre ciego.
  </div>
</div>
"""

HTML(pipeline_html)

   ---



   # SECCIÓN 2: Análisis Exploratorio Rápido

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Revisar valores nulos
missing_counts = df.isna().sum().sort_values(ascending=False)
print("Número de missing values por columna:")
print(missing_counts)

  Como veis, tenemos valores nulos en variables predictoras (thal, slope) y tambien en variable target (num), así como en ca, pero que nos da igual porque ya hemos dicho que la eliminaremos.



  ## Eliminación de pacientes sin información de la variable target "num":



  Fijaos que es diferente como tratamos los valores nulos NaN cuando estan en la variable target que cuando estan en los predictores. Si estan en los predictores podemos escoger que hacer con esos valores: si inputar, borrar el paciente, borrar la variable entera para todos los pacientes etc. (se explica más adelante). En canvio, si el valor nulo esta en la variable target, la observación/paciente no nos sirve de nada, ni para entrenar el modelo, ni para testearlo, ni para ninguna cosa. Tenemos que borrarlo.

In [ ]:
# Analizar variable objetivo 'num'
print("Distribución original de 'num':")
print(df['num'].value_counts().sort_index())

#Ved que si ponemos que no quite los NaN (dropna=False) nos los cuenta
print("Distribución original de 'num':")
print(df['num'].value_counts(dropna=False).sort_index())

In [ ]:
# Convertir a problema binario, ademas ya codificamos 0 como no enfermedad y 1 como enfermedad, no con strings, ya que si fuesen strings tendriamos que pasarlo a 0/1
df['target'] = (df['num'] > 0).astype(int)

#Veamos la distribución de la nueva variable target
print("\nDistribución binaria (target):")
print(df['target'].value_counts())
print(f"\nPrevalencia de enfermedad: {df['target'].mean()*100:.1f}%")



   Observación: El dataset está relativamente balanceado (~45% con enfermedad).



   Esto es favorable para el aprendizaje del modelo. Aunque no refleja la prevalencia real en screening poblacional de estenosis coronaria (sería mucho menor), el grupo de pacientes sobre los que se ha creado el dataset y tambien sobre el que se aplicará el modelo ya fueron seleccionados para cateterismo basándose en criterio clínico previo.

In [ ]:
df = df.dropna(subset=['num'])
print(f"\nDespués de eliminar pacientes sin 'num': {df.shape[0]} pacientes restantes.")

In [ ]:
# Estadísticas descriptivas
df.describe()

   Observaciones:



   - Variables con diferentes escalas (edad: 29-77, colesterol: 126-564)



   - Algunas variables son categóricas codificadas como números



   - Presencia de valores faltantes que requieren manejo

   ## Verificación de duplicados - Evitar data leakage



   Problema: Si el mismo paciente aparece múltiples veces en el dataset, podría



   terminar en AMBOS conjuntos (train y test), causando data leakage severo.



   Consecuencia: El modelo "vería" al paciente durante el entrenamiento y luego



   lo "evaluaría" en el test, inflando artificialmente las métricas.



   Solución: Verificar que cada fila representa un paciente único.

In [ ]:
# %%
# Verificar duplicados completos (filas idénticas)
duplicados_completos = df.duplicated().sum()

if duplicados_completos > 0:
    print(f"ALERTA: {duplicados_completos} filas duplicadas encontradas")
    df = df.drop_duplicates()
    print(f"Duplicados eliminados. Dataset: {df.shape[0]} pacientes")
else:
    print(f"OK: No hay duplicados ({df.shape[0]} pacientes únicos)")

# Verificar duplicados parciales basados en variables clave
columnas_clave = ['age', 'sex', 'trestbps', 'chol', 'thalach']
duplicados_parciales = df.duplicated(subset=columnas_clave).sum()

if duplicados_parciales > 0:
    print(f"ALERTA: {duplicados_parciales} posibles pacientes repetidos (verificar manualmente)")

  ### Eliminación de variables residuales que no queremos (ca y num)



  Como la información de la variable target original num ya la tenemos codificada en una nueva llamada target, a partir de ahora solo trabajaremos con target. Num la tenemos que borrar para evitar situaciones indeseadas como ponerla sin querer dentro de las variables predictores y generar data leakage de target en predictores indirectamente.



  Tambien tenemos que borrar la variable ca por la misma razón, porque es una variable que contiene información directa del proceso que queremos predecir y no la tendremos en el futuro.



  Para eliminarlas usaremos el metodo dataframe.drop().



  Este metodo se puede usar tanto para borrar columnas como filas (usando el nombre de columna como df.drop(columns=['ca','num'])) o por filas, donde le indicamos el indice de la fila que queremos borrar df.drop([2,5], axis=0)



  Fijaos que axis = 0 indica fila y axis = 1 indica columnas, igual que en la notación matemática para matrices i,j.

In [ ]:
# La función drop sirve tanto para eliminar filas como columnas concretas.
df = df.drop(columns=['ca', 'num'])
print(f"\nDespués de eliminar variables 'ca' y 'num': {df.shape[1]} variables restantes.")

   ---



   # SECCIÓN 3: Identificación de Variables Categóricas



   Miramos los tipos de datos del dataset para identificar qué variables son categóricas



   y necesitarán codificación. La transformación real (One-Hot Encoding) la aplicaremos



   en la Sección 6 (Parte 1), después del split, por las mismas razones que la imputación



   y el escalado: no queremos que el test set influya en nada que "aprendemos" de los datos. Los datos de test han de ser invisibles en la fase de entrenamiento.

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# Variables categóricas en el dataset
categorical_cols = ['cp', 'restecg', 'slope']
categorical_cols = [col for col in categorical_cols if col in df.columns]

print(f"Variables categóricas identificadas: {categorical_cols}")
for col in categorical_cols:
    print(f"  {col}: {df[col].nunique()} categorías - valores: {sorted(df[col].unique())}")

# La codificación (One-Hot Encoding) se aplica en la Sección 6 (Parte 1), tras el split.

   ---



   # SECCIÓN 4: Definición de Estrategias de Modelado



   Implementaremos DOS estrategias según disponibilidad de datos clínicos.



   Estrategia 1 — Modelo Básico: solo usa datos de consulta, ECG y prueba de esfuerzo.

   Es aplicable a cualquier paciente desde el primer momento. No requiere gammagrafía.



   Estrategia 2 — Modelo Completo: incluye también la gammagrafía de perfusión (thal).



   Solo aplicable cuando ya se dispone de ese resultado, es decir, en casos equívocos



   que ya han pasado por ese paso adicional.



   En ambas estrategias, la variable 'ca' está excluida por data leakage (se mide



   durante el propio cateterismo que queremos predecir).



   Las listas exactas de features las definimos en la Sección 6 (Parte 1), una vez que



   tengamos las columnas reales tras el One-Hot Encoding.

   ---



   # SECCIÓN 5: División Train/Test



   ## El problema fundamental del ML



   Si entrenamos y evaluamos en los mismos datos, estaremos estimando lo bien que lo hace el modelo sobre unos datos que conoce y recuerda. Por tanto el resultado será mucho más alto que en datos reales, que no haya visto nunca.



   Como darle a un estudiante las preguntas del examen antes del examen. No evaluas lo que ha entendido en el entrenamiendo, sino cuanto ha memorizado las preguntas.



   Cuando evaluamos la performance de nuestro modelo, lo que estamos haciendo es siempre intentar aproximar la performance *en producción* (en el mundo real). Como los resultados del mundo real no los tenemos nunca de antemano (ya que entonces no necesitariamos un modelo predictivo para nada) lo que hacemos es buscar diferentes metodos de aproximar esto.



   El primermanera de evaluar el modelo sería evaluarlo en los mismos datos que usamos para entrenarlo, en el train. Como ya hemos dicho, esta no sera una buena aproximación de la realidad, ya que el modelo se sabe mucho mejor los datos con los que ha sido entrenado de lo que se sabrá los de producción.



   La segunda opción es dividir los datos en train y test, entrenar y diseñar solo en train y usar los datos de test como datos ciegos, que el modelo no ha visto nunca, emulando que son datos de producción. Esta opción es mucho mejor, y nos da una estimación puntual de la performance en producción (puntual como un único valor) e.g. sensitividad = 0.9, especificidad = 0.7



   Hay técnicas más sofisticadas al respecto como la validación cruzada para estimar la performance del modelo en la realidad, que veremos también más adelante, que nos permiten ir más allá de una estimación puntual y nos permiten hacer una estimación más interesante, donde tenemos una serie de posibles valores para la performance, no solo podiendo sacar la media sino tambien la variabilidad de esta performance. Lo veremos más adelante



   ## Solución: Train/Test Split



   - Train set (80%): Entrenar el modelo



   - Test set (20%): Completamente oculto, simula pacientes nuevos



   Usar el 80% de los datos para entrenar y el 20 para testear es una convención, se podria usar cualquier proporcion deseada, cuidando que el modelo tenga suficiente datos en train como para aprender a clasificar el problema.



   El test set se usa UNA SOLA VEZ al final para estimar performance real.

In [ ]:
# %%
# Preparar datos para el split
# Usamos df directamente (sin codificar todavía las categóricas)
X = df.drop(columns=['target'])
y = df['target']

print(f"Dimensiones X: {X.shape}")
print(f"Columnas (antes del encoding): {list(X.columns)}")

# Train/Test Split (80/20) con estratificación para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=7,
    stratify=y
)

print(f"\nTrain: {X_train.shape[0]} pacientes ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} pacientes ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nVerificación de estratificación:")
print(f"  Enfermedad en Train:    {y_train.mean()*100:.1f}%")
print(f"  Enfermedad en Test:     {y_test.mean()*100:.1f}%")
print(f"  Enfermedad en Original: {y.mean()*100:.1f}%")




  ### Explicación del random_state y stratify:



  ¿Por qué random_state=42?



  El random_state es como fijar una "semilla" para el generador de números aleatorios. Sin él, cada vez que ejecutes el código obtendrías una división diferente de train/test (diferentes pacientes en cada conjunto), lo que haría imposible reproducir tus resultados o comparar experimentos. Al fijar random_state=42 (o cualquier número), garantizas que siempre obtendrás exactamente la misma división de datos, permitiendo que tú o cualquier persona pueda replicar tus resultados exactos. El número 42 es una convención popular (referencia a "La Guía del Autoestopista Intergaláctico"), pero podrías usar cualquier número.



  ¿Por qué stratify=y?



  La estratificación es crucial cuando tienes clases desbalanceadas. Imagina que en tu dataset original el 30% de pacientes tienen la enfermedad y el 70% están sanos. Sin estratificación, por puro azar podrías terminar con un conjunto de test donde solo el 20% tiene la enfermedad, o peor aún, donde el 45% la tiene. Esto distorsionaría completamente la evaluación de tu modelo.



  Al usar stratify=y, le dices a scikit-learn: "mantén la misma proporción de enfermos/sanos en train y test que existe en el dataset original". Si tienes 30% de enfermos en total, tendrás aproximadamente 30% en train Y 30% en test. Esto es especialmente importante en problemas médicos donde la clase minoritaria (los enfermos) es la más importante de detectar correctamente.

  ---



  # SECCIÓN 6: Preprocesamiento - Parte 1: One-Hot Encoding



  *Importante: Todo el preprocesamiento lo hacemos en train, DESPUÉS del split*



  ## ¿Qué es el One-Hot Encoding y por qué lo necesitamos?



  Los modelos de ML trabajan con números. Cuando una variable categórica está codificada



  como entero (cp: 1, 2, 3, 4), el modelo interpreta que 4 > 3 > 2 > 1, es decir, que



  existe un orden o jerarquía. Eso puede ser correcto para variables ordinales (como



  "gravedad: leve/moderada/severa"), pero no para variables nominales donde los valores



  son simplemente etiquetas sin orden real.



  El One-Hot Encoding resuelve esto convirtiendo cada categoría en una columna binaria



  independiente (0 o 1). El modelo ya no ve jerarquías, solo presencia o ausencia.



  Ejemplo concreto: imagina una variable "tipo_dolor" con valores 1, 2, 3, 4.



  Tras el encoding, desaparece esa columna y aparecen tres nuevas (drop_first=True

  elimina una columna — la correspondiente a la categoría 1 — porque es redundante:

  si tipo_dolor_2=0, tipo_dolor_3=0 y tipo_dolor_4=0, el modelo ya sabe que es tipo 1.

  Mantener las 4 columnas daría información duplicada):



    Antes:  tipo_dolor = 3



    Después: tipo_dolor_2=0, tipo_dolor_3=1, tipo_dolor_4=0



  El modelo ahora sabe que este paciente tiene "tipo 3" sin asumir que 3 > 2 o que



  tipo 4 es "el doble" que tipo 2.



  Cuándo usarlo: variables categóricas nominales (sin orden inherente), especialmente



  en algoritmos basados en distancias (KNN) o lineales (regresión logística).



  Cuándo NO usarlo: variables ordinales donde el orden sí importa (bajo/medio/alto),



  o cuando hay demasiadas categorías (alta cardinalidad), en cuyo caso hay alternativas



  como target encoding.



  ## ¿Por qué aplicarlo después del split?



  Con get_dummies no se "aprende" ninguna estadística del dataset (a diferencia de la



  media para imputación o la std para escalado). Sin embargo, el problema práctico es



  que si una categoría aparece solo en test y no en train, se generarían columnas



  distintas en cada set, rompiendo el modelo en producción.



  La solución es aplicarlo por separado a train y test, y luego alinear las columnas



  con reindex. Si alguna categoría falta en test, se rellena con 0 (no apareció = no activa).



  En este dataset el riesgo es mínimo (todos los valores de cp, restecg y slope aparecen



  en ambos sets), pero la práctica correcta es seguir siempre el pipeline consistente.

In [ ]:
# %%
# Aplicar One-Hot Encoding DESPUÉS del split (por separado en train y test)
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=categorical_cols, drop_first=True, dtype=int)

# Alinear columnas: si alguna categoría de las que hay en test no aparecia en train, se añade con valor 0 ( o el que decidamos)
# Esto garantiza que train y test tienen exactamente las mismas columnas
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Shape X_train tras encoding: {X_train.shape}")
print(f"Shape X_test  tras encoding: {X_test.shape}")
nuevas = [c for c in X_train.columns if any(cat in c for cat in categorical_cols) and '_' in c]
print(f"\nColumnas nuevas generadas: {nuevas}")




In [ ]:
# %%
# Definir features para cada estrategia ahora que tenemos las columnas exactas
all_features = list(X_train.columns)

# ESTRATEGIA 1: sin 'thal' (gammagrafía de perfusión)
basic_features    = [col for col in all_features if col != 'thal']
# ESTRATEGIA 2: con 'thal'
complete_features = all_features.copy()

print(f"ESTRATEGIA 1 — Modelo Básico:   {len(basic_features)} variables")
print(f"ESTRATEGIA 2 — Modelo Completo: {len(complete_features)} variables")
print(f"Variable adicional en Completo: {[c for c in complete_features if c not in basic_features]}")
print(f"\nLista básico: {basic_features}")




   ---



   # SECCIÓN 6: Preprocesamiento - Parte 2: Valores Faltantes



   *Importante: También DESPUÉS del split para evitar data leakage*



   ## Por qué DESPUÉS del split



   Algunos modelos ML no pueden trabajar con valores NaN. Usaremos mediana para imputación (robusta a outliers).



   Flujo incorrecto:



   1. Calcular mediana con todos los datos (train + test)



   2. Imputar valores faltantes



   3. Hacer train/test split



   Problema: La mediana fue calculada usando información del test set, causando data leakage.



   Flujo correcto:



   1. Hacer train/test split primero



   2. Calcular mediana SOLO con datos de train



   3. Aplicar esa mediana a train



   4. Aplicar la MISMA mediana (del train) al test



   Analogía clínica: Es como calcular valores de referencia de un test diagnóstico.



   Los valores de referencia se calculan con la población de desarrollo, NO incluyendo



   la población de validación. Luego se aplican esos mismos valores a la validación.

In [ ]:
# %%
columns_with_nulls = X_train.columns[X_train.isnull().any()].tolist()

if len(columns_with_nulls) > 0:
    print(f"Variables con valores nulos: {columns_with_nulls}")
    print(f"\nValores nulos antes de imputación:")
    print(f"  Train: {X_train.isnull().sum().sum()}")
    print(f"  Test: {X_test.isnull().sum().sum()}")
    
    # Imputación con mediana calculada SOLO del train
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train)  # Aprende SOLO de train
    
    X_train = pd.DataFrame(
        imputer.transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )
    
    X_test = pd.DataFrame(
        imputer.transform(X_test),  # Usa medianas del train
        columns=X_test.columns,
        index=X_test.index
    )
    
    print(f"\nValores nulos después de imputación:")
    print(f"  Train: {X_train.isnull().sum().sum()}")
    print(f"  Test: {X_test.isnull().sum().sum()}")
else:
    print("No hay valores nulos en train ni test")




   ---



   # SECCIÓN 6: Preprocesamiento - Parte 3: Escalado



   *Importante: También aplicar DESPUÉS del split para evitar data leakage*



   ## Por qué escalar



   Las variables tienen escalas muy diferentes (edad: 29-77, colesterol: 126-564).



   Algunos algoritmos (Logistic Regression, KNN) son sensibles a estas diferencias, porque están basados internamente en métricas de "distancia".



   Ejemplo concreto con KNN: si calculamos la distancia entre dos pacientes sin escalar,

   la diferencia de colesterol (ej. 200 vs 250 = diferencia de 50) domina completamente

   sobre la diferencia de sexo (0 vs 1 = diferencia de 1) o de fbs (0 vs 1 = diferencia de 1).

   El modelo pensaría que "pacientes similares" significa "colesterol parecido", ignorando

   las demás variables. Tras escalar, todas las variables contribuyen por igual.



   ## StandardScaler



   Transforma a media=0, desviación estándar=1.



   ## Por qué DESPUÉS del split



   Mismo principio que en la imputación (Parte 2): calcular media/std antes del split

   usaría información del test set — data leakage. El scaler se entrena SOLO con X_train

   y luego se aplica a X_test usando esas estadísticas.

In [ ]:
# %%
scaler = StandardScaler()

# Fit en train (aprende media y std)
X_train_scaled = scaler.fit_transform(X_train)
# Transform en test (usa estadísticas del train)
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.describe()



  Observación: Efectivamente nuestras variables continuas ahora tienen media 0 y std 1

   ---



   # SECCIÓN 8: Modelo baseline / modelo dummy o tonto



   ## Por qué necesitamos un baseline



   Antes de construir modelos complejos, necesitamos un punto de referencia:



   el enfoque más simple posible.



   DummyClassifier con la estratégia most_frequent siempre predice la clase más frecuente.

   Es nuestro "mínimo a superar".



   ¿Por qué su AUC es exactamente 0.5?

   Porque asigna la misma probabilidad a todos los pacientes — no discrimina en absoluto.

   Al ordenar pacientes de mayor a menor probabilidad, los enfermos y sanos están

   mezclados al azar → la curva ROC sigue la diagonal → AUC = 0.5.

   Un modelo que no supere AUC = 0.5 predice PEOR que el azar.

In [ ]:
# %%
dummy_clf = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_clf.fit(X_train_scaled, y_train)

most_frequent_class = dummy_clf.classes_[np.argmax(dummy_clf.class_prior_)]

# Un modelo que siempre predice la clase más frecuente tiene AUC = 0.5
# porque sus probabilidades son idénticas para todos los pacientes (sin poder discriminar)
dummy_train_acc = accuracy_score(y_train, dummy_clf.predict(X_train_scaled))
dummy_test_acc  = accuracy_score(y_test,  dummy_clf.predict(X_test_scaled))

print(f"BASELINE MODEL (DummyClassifier — siempre predice clase más frecuente)")
print(f"  Train Accuracy: {dummy_train_acc*100:.2f}%  |  Test Accuracy: {dummy_test_acc*100:.2f}%")
print(f"  Predicción: siempre {'Enfermedad' if most_frequent_class == 1 else 'Sin Enfermedad'}")
print(f"\n  AUC del modelo dummy = 0.500 (sin capacidad discriminativa)")
print(f"  Cualquier modelo útil debe superar AUC = 0.5")




---

# Sección 9: Entendiendo la evaluación de performance

Ya sabemos que hacer el split train/test para evaluar el modelo en datos que no ha visto
(ver Sección 5) es una de las maneras de estimar la performance de nuestro modelo. Aquí
formalizamos las tres maneras de estimar esta performance en producción y justificamos por
qué la métrica AUC es la más adecuada para nuestro caso.

## Tres formas de evaluar un modelo

**1. Estimación en train**
Entrenar en unos datos y evaluar en los mismos. El modelo se sabe los datos de entrenamiento
mejor de lo que se sabrá los de producción, por tanto Train AUC sobreestima la performance real.

**2. Estimación en test**
Entrenamos en datos de train y evaluamos en datos diferentes (test). La performance en ese test
se parece más a la realidad, pero es una única estimación puntual con varianza alta si el test
set es pequeño — dependiendo de cómo hagamos la división, el valor de performance puede depender
más de *qué* datos usamos para el test que de lo bien que lo haga el modelo.

**3. Estimación en cross-validation**
La validación cruzada divide los datos de train en *n* grupos iterativamente, entrena en *n-1*
y valida en el grupo restante. Genera *n* estimaciones de AUC, dándonos media y variabilidad —
mucho más robusto que una única estimación puntual.

---

Durante el desarrollo usamos **únicamente los datos de train** (métodos 1 y 3) para no
"contaminar" el test set. Usamos AUC porque, a diferencia de la accuracy, es independiente
del threshold de decisión: nos dice cuán bien discrimina el modelo para todos los posibles
umbrales a la vez.

El mismo principio se aplica a otras métricas y parámetros: sensibilidad, especificidad,
PPV, el threshold óptimo, los hiperparámetros del modelo — todo se estima con CV sobre los
datos de train, nunca sobre el test set.

In [ ]:
# %%
# Demostración con Random Forest — tres formas de estimar la performance
demo_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
demo_model.fit(X_train_scaled, y_train)

# A) Train AUC
train_auc_demo = roc_auc_score(y_train, demo_model.predict_proba(X_train_scaled)[:, 1])

# B) Test AUC
test_auc_demo = roc_auc_score(y_test, demo_model.predict_proba(X_test_scaled)[:, 1])

# C) Cross-Validation AUC (solo sobre train, 10 folds)
cv_scores = cross_val_score(demo_model, X_train_scaled, y_train, cv=10, scoring='roc_auc')

print("TRES FORMAS DE EVALUAR UN MODELO (usando AUC, no accuracy):\n")
print(f"A) TRAIN AUC: {train_auc_demo:.3f}")
print("   El modelo 've' estos datos durante entrenamiento")
print("   AUC alta en train NO garantiza funcionar en pacientes nuevos\n")

print(f"B) TEST AUC: {test_auc_demo:.3f}")
print("   El modelo NUNCA vio estos datos")
print("   Estima performance real (pero solo una medición puntual)\n")

print(f"C) CROSS-VALIDATION AUC (10-Fold):")
print(f"   AUC por fold: {[f'{s:.3f}' for s in cv_scores]}")
print(f"   Media: {cv_scores.mean():.3f} (± {cv_scores.std():.3f})")
print("   Usa solo training data (divide en 10 folds los datos de train)")
print("   Da múltiples estimaciones — más robusto que un único test split")
print("   Test set permanece intacto")
print()
print("OBSERVA: Train AUC > CV AUC — la sección siguiente explica exactamente por qué.")




In [ ]:
# %%

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(
    cv_scores,
    bins="auto",
    kde=True,
    color="steelblue",
    edgecolor="white",
    alpha=0.75,
    ax=ax,
)

ax.axvline(cv_scores.mean(), color="tomato", lw=1.8, ls="--", label=f"Mean AUC = {cv_scores.mean():.3f}")
ax.axvline(np.median(cv_scores), color="darkorange", lw=1.8, ls=":", label=f"Median AUC = {np.median(cv_scores):.3f}")
ax.axvline(0.5, color="gray", lw=1.5, ls=":", alpha=0.6, label="Azar (AUC = 0.5)")

ax.set_xlabel("AUC", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("CV AUC Distribution (10-Fold)", fontsize=13)
ax.legend(frameon=False)
sns.despine()

plt.tight_layout()
plt.savefig("cv_auc_dist.png", dpi=150)
plt.show()


  ---



  # SECCIÓN 9.1: Bias-Variance Tradeoff — El dilema central del ML



  El demo de arriba probablemente muestra Train AUC bastante más alta que CV AUC. Antes de seguir entrenando modelos, es importante entender por qué ocurre eso y qué significa. Es el concepto más importante de todo el aprendizaje supervisado: el trade-off entre bias y varianza.



  ## ¿Qué es el bias (sesgo)?



  El bias es el error que comete un modelo por ser demasiado simple para capturar la complejidad real del problema. Un modelo con mucho bias hace suposiciones demasiado rígidas y "no aprende suficiente" de los datos.



  Ejemplo clínico: Imagina predecir enfermedad coronaria usando solo una regla: "si edad > 55, enfermedad = sí". Muchos pacientes jóvenes enfermos quedarían sin detectar. Eso es alto bias — el modelo no tiene arquitectura con complejidad suficiente como para representar la realidad.



  En ML: Logistic Regression tiene más bias que Random Forest porque asume que la frontera de decisión entre sanos y enfermos es una línea recta en el espacio de features. Si la realidad es más compleja (curvilínea, con interacciones entre variables), el modelo nunca lo captará, por más datos que le des.



  ## ¿Qué es la varianza?



  La varianza es el error que introduce un modelo por ser demasiado complejo: aprende no solo el patrón real sino también parte del "ruido" particular de los datos de entrenamiento. Funciona perfectamente en train pero puede fallar en datos nuevos si el modelado no se hace con cuidado.



  Ejemplo clínico: Un Decision Tree sin límite de profundidad puede crear reglas tan específicas como "si thalach = 142 Y oldpeak = 1.4 Y sexo = 0 → enfermedad". Memoriza a los 242 pacientes de train. Cuando llega un paciente nuevo con thalach = 143, el árbol no sabe qué hacer. Eso es alta varianza: aprendió de memoria casos concretos, no el patrón general.



  Lo detectamos cuando Train AUC >> CV AUC (gap grande entre ambas).



  ## El trade-off



  No podemos minimizar bias y varianza simultáneamente. Hacer el modelo más complejo reduce bias pero aumenta varianza, y viceversa. El objetivo es encontrar el equilibrio:



  ```



  Error total = Bias² + Varianza + Ruido irreducible (no eliminable)



  ```



 | Modelo | Bias | Varianza | Intuición |
 |---|---|---|---|
 | Logistic Regression | Alto | Bajo | Simple y estable, puede no capturar complejidad |
 | Decision Tree (sin límite) | Bajo | Muy alto | Flexible pero tiende a memorizar |
 | Decision Tree (max_depth=5) | Medio | Medio | Limitamos profundidad para controlar varianza |
 | Random Forest | Bajo | Muy bajo | 100 árboles → su varianza individual se promedia |
 | KNN (K=5) | Medio | Medio | K pequeño = más varianza; K grande = más bias |





   ---



   # SECCIÓN 10: ESTRATEGIA 1 - Modelo Básico (Sin Gammagrafía)



   Entrenamos modelos usando solo variables de consulta + prueba esfuerzo.



   Aplicable a TODOS los pacientes.

In [ ]:
# Preparar datos para Modelo Básico
X_train_basic = X_train_scaled[basic_features]
X_test_basic = X_test_scaled[basic_features]

print(f"ESTRATEGIA 1: MODELO BÁSICO")
print(f"Variables usadas: {len(basic_features)} de {len(complete_features)} totales")
print(f"Excluidas: thal (gammagrafía), ca (cateterismo)")
print(f"Train: {X_train_basic.shape}, Test: {X_test_basic.shape}")

  ### Modelo 1: Regresión Logística



  Supuesto clave: Asume que existe una frontera lineal que separa sanos de enfermos.

  En 2 variables imagina una línea recta; en más variables, un plano o hiperplano.

  Ejemplo: "si 0.3 × edad + 0.5 × oldpeak > umbral → enfermo". Siempre es una suma

  ponderada de variables, nunca captura relaciones tipo "la edad importa más cuando

  el colesterol es alto" (interacciones) ni curvas. Si la realidad es más compleja,

  el modelo nunca lo aprenderá independientemente de cuántos datos le des.



  ¿Por qué usarla entonces?



  1. Es muy interpretable: el coeficiente de cada variable dice directamente cuánto y en qué dirección contribuye al riesgo (similar a una regresión logística clásica epidemiológica)



  2. Entrenamiento rapidísimo y numéricamente estable



  3. Excelente punto de referencia: si un modelo más complejo no la supera claramente, igual de sencillo usar ésta



  Señal de bias alto: tanto Train AUC como CV AUC son moderados o bajos, y el gap entre ellas es pequeño. Esto indica que el modelo no puede aprender más aunque le des más datos — su arquitectura es demasiado simple para el problema. A diferencia del overfitting (Train alta, CV baja), aquí ambas son mediocres.

In [ ]:
# %%
# Modelo 1: Logistic Regression (Básico)
log_reg_basic = LogisticRegression(random_state=42, max_iter=1000)
log_reg_basic.fit(X_train_basic, y_train)

log_reg_basic_train = roc_auc_score(y_train, log_reg_basic.predict_proba(X_train_basic)[:, 1])
log_reg_basic_cv    = cross_val_score(log_reg_basic, X_train_basic, y_train, cv=5, scoring='roc_auc')
log_reg_basic_test  = roc_auc_score(y_test, log_reg_basic.predict_proba(X_test_basic)[:, 1])

print("MODELO 1: LOGISTIC REGRESSION (Básico)")
print(f"Train AUC: {log_reg_basic_train:.3f} | CV AUC: {log_reg_basic_cv.mean():.3f}±{log_reg_basic_cv.std():.3f} | Test AUC: {log_reg_basic_test:.3f} | Gap: {log_reg_basic_train - log_reg_basic_cv.mean():.3f}")




  ### Modelo 2: Decision Tree



  Mecanismo: Construye una serie de preguntas binarias ("¿thalach > 150?", "¿oldpeak > 2.3?") que forman un árbol. Cada hoja del árbol corresponde a una decisión final.



  Supuesto clave: Ninguno sobre la distribución de los datos. Puede dividir el espacio de features de forma arbitrariamente compleja si se lo permitimos.



  El problema del overfitting: Sin `max_depth`, el árbol crece hasta memorizar cada paciente de entrenamiento individualmente — alta varianza. Usamos `max_depth=5` para limitarlo. Este parámetro es un dial de varianza: más profundidad = más varianza; menos profundidad = más bias.



  Señal de varianza alta: Train AUC >> CV AUC. Si pruebas a usar `max_depth=None` (sin límite), verás que Train AUC llega a 1.0 pero CV AUC cae. Eso es overfitting puro.

In [ ]:
# %%
# Modelo 2: Decision Tree (Básico)
tree_basic = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_basic.fit(X_train_basic, y_train)

tree_basic_train = roc_auc_score(y_train, tree_basic.predict_proba(X_train_basic)[:, 1])
tree_basic_cv    = cross_val_score(tree_basic, X_train_basic, y_train, cv=5, scoring='roc_auc')
tree_basic_test  = roc_auc_score(y_test, tree_basic.predict_proba(X_test_basic)[:, 1])

print("MODELO 2: DECISION TREE (Básico)")
print(f"Train AUC: {tree_basic_train:.3f} | CV AUC: {tree_basic_cv.mean():.3f}±{tree_basic_cv.std():.3f} | Test AUC: {tree_basic_test:.3f} | Gap: {tree_basic_train - tree_basic_cv.mean():.3f}")




  ### Modelo 3: Random Forest



  Mecanismo: Entrena 100 árboles de decisión independientes. Cada árbol se entrena en una muestra aleatoria con reemplazo de los datos (bootstrap) y en cada división solo puede usar un subconjunto aleatorio de features. La predicción final es la votación mayoritaria de todos los árboles.



  ¿Por qué funciona mejor que un árbol solo? Cada árbol individual tiene alta varianza (sobreajusta a su muestra), pero al promediar 100 votos los errores individuales se cancelan. Si un árbol se equivoca porque vio una muestra rara, los otros 99 compensan. Esto reduce la varianza a casi cero sin aumentar el bias: es el bagging que vimos en la sección anterior, aplicado en la práctica.



  Resultado esperado: Gap Train-CV mucho menor que en el Decision Tree individual, y CV AUC superior. Si no ocurre así, sería una señal de que el problema tiene muy alta irreducibilidad (mucho ruido intrínseco).

In [ ]:
# %%
# Modelo 3: Random Forest (Básico)
rf_basic = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_basic.fit(X_train_basic, y_train)

rf_basic_train = roc_auc_score(y_train, rf_basic.predict_proba(X_train_basic)[:, 1])
rf_basic_cv    = cross_val_score(rf_basic, X_train_basic, y_train, cv=5, scoring='roc_auc')
rf_basic_test  = roc_auc_score(y_test, rf_basic.predict_proba(X_test_basic)[:, 1])

print("MODELO 3: RANDOM FOREST (Básico)")
print(f"Train AUC: {rf_basic_train:.3f} | CV AUC: {rf_basic_cv.mean():.3f}±{rf_basic_cv.std():.3f} | Test AUC: {rf_basic_test:.3f} | Gap: {rf_basic_train - rf_basic_cv.mean():.3f}")




  ### Modelo 4: K-Nearest Neighbors (KNN)



  Mecanismo: Para predecir si un nuevo paciente está enfermo, busca los K=5 pacientes más "cercanos" en el espacio de features por distancia euclidiana, y decide por votación mayoritaria entre ellos.



  Supuesto clave: Pacientes con características clínicas similares (edad, colesterol, thalach parecidos...) deberían tener el mismo diagnóstico. La "similitud" se mide geométricamente como distancia en el espacio de features — por eso el escalado (Sección 6 Parte 3) es absolutamente crítico aquí: sin escalar, el colesterol (rango 126-564) dominaría completamente sobre el sexo (0-1), haciendo que "cercano" signifique "colesterol parecido" más que "perfil clínico similar".



  Sobre la elección de K=5:

  K pequeño (ej. K=1) → el modelo es muy sensible a cada paciente individual, alta varianza.

  K grande (ej. K=50) → el modelo promedia muchos pacientes, pierde sensibilidad local, alto bias.

  K=5 es una convención de partida razonable para datasets de tamaño medio. En un proyecto real

  se debería optimizar K mediante GridSearchCV, que se cubre en secciones avanzadas.



  Limitaciones:



  - Lento en producción: debe calcular distancias a todos los pacientes de train



  - Sensible a features irrelevantes que distorsionan las distancias



  - Poca interpretabilidad: no hay una fórmula, solo "se parece a estos 5 pacientes"

In [ ]:
# %%
# Modelo 4: KNN (Básico)
knn_basic = KNeighborsClassifier(n_neighbors=5)
knn_basic.fit(X_train_basic, y_train)

knn_basic_train = roc_auc_score(y_train, knn_basic.predict_proba(X_train_basic)[:, 1])
knn_basic_cv    = cross_val_score(knn_basic, X_train_basic, y_train, cv=5, scoring='roc_auc')
knn_basic_test  = roc_auc_score(y_test, knn_basic.predict_proba(X_test_basic)[:, 1])

print("MODELO 4: K-NEAREST NEIGHBORS (Básico)")
print(f"Train AUC: {knn_basic_train:.3f} | CV AUC: {knn_basic_cv.mean():.3f}±{knn_basic_cv.std():.3f} | Test AUC: {knn_basic_test:.3f} | Gap: {knn_basic_train - knn_basic_cv.mean():.3f}")




   ---



   # SECCIÓN 11: ESTRATEGIA 2 - Modelo Completo (Con Gammagrafía)



   Entrenamos modelos incluyendo resultado de gammagrafía.



   Solo aplicable a pacientes que ya tienen gammagrafía.

In [ ]:
# %%
print(f"ESTRATEGIA 2: MODELO COMPLETO")
print(f"Variables usadas: {len(complete_features)}")
print(f"Incluye: thal (gammagrafía)")
print(f"Excluye: ca (cateterismo - data leakage)")
print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")




In [ ]:
# %%
# Modelo 1: Logistic Regression (Completo)
log_reg_complete = LogisticRegression(random_state=42, max_iter=1000)
log_reg_complete.fit(X_train_scaled, y_train)

log_reg_complete_train = roc_auc_score(y_train, log_reg_complete.predict_proba(X_train_scaled)[:, 1])
log_reg_complete_cv    = cross_val_score(log_reg_complete, X_train_scaled, y_train, cv=5, scoring='roc_auc')
log_reg_complete_test  = roc_auc_score(y_test, log_reg_complete.predict_proba(X_test_scaled)[:, 1])

print("MODELO 1: LOGISTIC REGRESSION (Completo)")
print(f"Train AUC: {log_reg_complete_train:.3f} | CV AUC: {log_reg_complete_cv.mean():.3f}±{log_reg_complete_cv.std():.3f} | Test AUC: {log_reg_complete_test:.3f} | Gap: {log_reg_complete_train - log_reg_complete_cv.mean():.3f}")




In [ ]:
# %%
# Modelo 2: Decision Tree (Completo)
tree_complete = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_complete.fit(X_train_scaled, y_train)

tree_complete_train = roc_auc_score(y_train, tree_complete.predict_proba(X_train_scaled)[:, 1])
tree_complete_cv    = cross_val_score(tree_complete, X_train_scaled, y_train, cv=5, scoring='roc_auc')
tree_complete_test  = roc_auc_score(y_test, tree_complete.predict_proba(X_test_scaled)[:, 1])

print("MODELO 2: DECISION TREE (Completo)")
print(f"Train AUC: {tree_complete_train:.3f} | CV AUC: {tree_complete_cv.mean():.3f}±{tree_complete_cv.std():.3f} | Test AUC: {tree_complete_test:.3f} | Gap: {tree_complete_train - tree_complete_cv.mean():.3f}")




In [ ]:
# %%
# Modelo 3: Random Forest (Completo)
rf_complete = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_complete.fit(X_train_scaled, y_train)

rf_complete_train = roc_auc_score(y_train, rf_complete.predict_proba(X_train_scaled)[:, 1])
rf_complete_cv    = cross_val_score(rf_complete, X_train_scaled, y_train, cv=5, scoring='roc_auc')
rf_complete_test  = roc_auc_score(y_test, rf_complete.predict_proba(X_test_scaled)[:, 1])

print("MODELO 3: RANDOM FOREST (Completo)")
print(f"Train AUC: {rf_complete_train:.3f} | CV AUC: {rf_complete_cv.mean():.3f}±{rf_complete_cv.std():.3f} | Test AUC: {rf_complete_test:.3f} | Gap: {rf_complete_train - rf_complete_cv.mean():.3f}")




In [ ]:
# %%
# Modelo 4: KNN (Completo)
knn_complete = KNeighborsClassifier(n_neighbors=5)
knn_complete.fit(X_train_scaled, y_train)

knn_complete_train = roc_auc_score(y_train, knn_complete.predict_proba(X_train_scaled)[:, 1])
knn_complete_cv    = cross_val_score(knn_complete, X_train_scaled, y_train, cv=5, scoring='roc_auc')
knn_complete_test  = roc_auc_score(y_test, knn_complete.predict_proba(X_test_scaled)[:, 1])

print("MODELO 4: K-NEAREST NEIGHBORS (Completo)")
print(f"Train AUC: {knn_complete_train:.3f} | CV AUC: {knn_complete_cv.mean():.3f}±{knn_complete_cv.std():.3f} | Test AUC: {knn_complete_test:.3f} | Gap: {knn_complete_train - knn_complete_cv.mean():.3f}")




   ---



   # SECCIÓN 12: Comparación de Estrategias

In [ ]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# Tabla comparativa — AUC (no accuracy: es threshold-dependent)
#
# IMPORTANTE: la columna Test AUC se muestra SOLO con fines pedagógicos,
# para que veáis cómo se relaciona con CV AUC. En la práctica, la selección
# del modelo ganador se debe hacer EXCLUSIVAMENTE con CV AUC — el test set
# se toca una sola vez al final, para el modelo ya seleccionado.
# Si usásemos Test AUC para elegir el modelo, estaríamos ajustando la decisión
# a esos datos concretos y la estimación ya no sería imparcial.
# ─────────────────────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Modelo':       ['LogReg', 'Tree', 'RF', 'KNN',
                     'LogReg', 'Tree', 'RF', 'KNN'],
    'Estrategia':   ['Básico'] * 4 + ['Completo'] * 4,
    'Train AUC':    [log_reg_basic_train,  tree_basic_train,  rf_basic_train,  knn_basic_train,
                     log_reg_complete_train, tree_complete_train, rf_complete_train, knn_complete_train],
    'CV AUC':       [log_reg_basic_cv.mean(),  tree_basic_cv.mean(),  rf_basic_cv.mean(),  knn_basic_cv.mean(),
                     log_reg_complete_cv.mean(), tree_complete_cv.mean(), rf_complete_cv.mean(), knn_complete_cv.mean()],
    'CV ± std':     [log_reg_basic_cv.std(),  tree_basic_cv.std(),  rf_basic_cv.std(),  knn_basic_cv.std(),
                     log_reg_complete_cv.std(), tree_complete_cv.std(), rf_complete_cv.std(), knn_complete_cv.std()],
    'Test AUC':     [log_reg_basic_test,  tree_basic_test,  rf_basic_test,  knn_basic_test,
                     log_reg_complete_test, tree_complete_test, rf_complete_test, knn_complete_test],
    'Gap Train-CV': [log_reg_basic_train  - log_reg_basic_cv.mean(),
                     tree_basic_train     - tree_basic_cv.mean(),
                     rf_basic_train       - rf_basic_cv.mean(),
                     knn_basic_train      - knn_basic_cv.mean(),
                     log_reg_complete_train - log_reg_complete_cv.mean(),
                     tree_complete_train    - tree_complete_cv.mean(),
                     rf_complete_train      - rf_complete_cv.mean(),
                     knn_complete_train     - knn_complete_cv.mean()],
})

comparison = comparison.round(3)
print("COMPARACIÓN COMPLETA: BÁSICO vs COMPLETO  (métrica: AUC)")
print()
print("Cómo leer esta tabla:")
print("  - CV AUC:      la métrica principal para seleccionar modelo (honesta, solo usa train)")
print("  - Train AUC:   siempre más alta; si está muy por encima de CV AUC → overfitting")
print("  - Test AUC:    mostrado para aprendizaje; en práctica real se miraría solo al final")
print("  - Gap Train-CV: > 0.10 indica overfitting notable")
print("  - CV ± std:    variabilidad entre folds; alta std = resultados poco estables")
print()
print("Busca: CV AUC máxima + Gap pequeño + std baja")
comparison



In [ ]:
# %%
# Gráfico comparativo: 3 barras por modelo (Train / CV / Test AUC)
# ─────────────────────────────────────────────────────────────────
# Train AUC (azul claro): siempre más alta — el modelo vio estos datos
# CV AUC (verde):         la métrica honesta para selección de modelo
# Test AUC (naranja):     evaluación final — debería alinearse con CV
# Gap grande Train vs CV: señal de overfitting (alta varianza)

basic_models    = comparison[comparison['Estrategia'] == 'Básico'].reset_index(drop=True)
complete_models = comparison[comparison['Estrategia'] == 'Completo'].reset_index(drop=True)

x     = np.arange(4)
width = 0.22

model_labels = ['LogReg', 'Tree', 'RF', 'KNN']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df_strat, title in zip(
        axes,
        [basic_models, complete_models],
        ['Estrategia Básica (sin gammagrafía)', 'Estrategia Completa (con gammagrafía)']):

    trains = df_strat['Train AUC'].values
    cvs    = df_strat['CV AUC'].values
    tests  = df_strat['Test AUC'].values
    stds   = df_strat['CV ± std'].values

    ax.bar(x - width, trains, width, label='Train AUC', color='steelblue',      alpha=0.55)
    ax.bar(x,         cvs,    width, label='CV AUC',    color='mediumseagreen', alpha=0.85,
           yerr=stds, capsize=4, error_kw={'elinewidth': 1.5, 'alpha': 0.7})
    ax.bar(x + width, tests,  width, label='Test AUC',  color='darkorange',     alpha=0.85)

    ax.axhline(y=0.5, color='tomato', linestyle='--', linewidth=2, alpha=0.7,
               label='Azar (AUC = 0.5)')

    ax.set_ylabel('AUC', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels)
    ax.legend(framealpha=0.9, loc='lower right', fontsize=9)
    ax.set_ylim([0.4, 1.05])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Comparación de modelos — Train / CV / Test AUC', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


  Observaciones:



  - Todos los modelos superan AUC = 0.5 (azar) — el ML aporta valor real en este problema



  - Train AUC siempre es la más alta porque el modelo vio esos datos durante el entrenamiento.



  - CV AUC es la métrica honesta para seleccionar el mejor modelo: nos da múltiples mediciones en lugar de una única estimación puntual y además suele ser más conservadora que la performance en test, pareciendose más a la realidad



  - Test AUC debería alinearse con CV AUC. Si se aleja mucho, puede ser que el test es pequeño (varianza alta).



  - Los modelos Completos (con gammagrafía) superan a los Básicos en todos los algoritmos. La diferencia de AUC entre estrategias cuantifica el valor diagnóstico de añadir la gammagrafía de perfusión al proceso de decisión.



  - Random Forest muestra el mejor balance en ambas estrategias: alto AUC y Gap pequeño. El bagging cancela la varianza individual de cada árbol sin aumentar el bias.


  A partir de este momento continuamos con el Random Forest



   ---



   # SECCIÓN 13: Métricas Clínicas



   ## Métricas que importan en medicina:



   Como me habreis oido decir antes, la accuracy no lo es todo, puesto que si el coste monetario o humano de un FP y un FN no son parecidos, no nos ayuda a encontrar un buen compromiso, de la misma manera que también nos engaña si predecimos sobre enfermedades con muy poca prevalencia (nuestros datos estan muy desbalanceados). La metrica que hemos estado usando hasta ahora, AUC, tampoco nos sirve como metrica general de performance, ya que esta es threshold acnostic, y no nos da un numero quantificable de como de bien lo hará el modelo en los pacientes. Aquí aparecen metricas clinicas como sensitivity o specificity.



   Sensitivity (Recall): De todos los enfermos, ¿cuántos detectamos?



   - En medicina: NO queremos perder casos de enfermedad (priorizar Sensitivity)



   Specificity: De todos los sanos, ¿cuántos identificamos correctamente?



   - Evita procedimientos innecesarios



   PPV (Precision): Si predecimos "enfermedad", ¿qué probabilidad de estar en lo cierto?



   NPV: Si predecimos "sano", ¿qué probabilidad de estar en lo cierto?



   Evaluamos SOLO EN TEST SET (una única vez)

In [ ]:
# %%
def clinical_metrics(y_true, y_pred, model_name):
    """Calcula métricas clínicas relevantes"""
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    
    print(f"\nMÉTRICAS CLÍNICAS: {model_name}")
    print(f"{'='*70}")
    print(f"\nConfusion Matrix:")
    print(f"  TN: {TN} (sanos identificados correctamente)")
    print(f"  FP: {FP} (sanos clasificados como enfermos - cateterismo innecesario)")
    print(f"  FN: {FN} (enfermos clasificados como sanos - PELIGROSO)")
    print(f"  TP: {TP} (enfermos identificados correctamente)")
    
    print(f"\nMÉTRICAS:")
    print(f"  Accuracy: {accuracy*100:.1f}%")
    print(f"  Sensitivity (Recall): {sensitivity*100:.1f}% - Detecta {sensitivity*100:.0f}% de enfermos")
    print(f"  Specificity: {specificity*100:.1f}% - Identifica {specificity*100:.0f}% de sanos")
    print(f"  PPV (Precision): {ppv*100:.1f}% - Si predice enfermedad, {ppv*100:.0f}% correcto")
    print(f"  NPV: {npv*100:.1f}% - Si predice sano, {npv*100:.0f}% correcto")
    
    print(f"\nCASOS PERDIDOS: {FN} pacientes con enfermedad NO detectados")
    
    return {
        'Model': model_name,
        'Sensitivity': sensitivity * 100,
        'Specificity': specificity * 100,
        'PPV': ppv * 100,
        'NPV': npv * 100,
        'Accuracy': accuracy * 100,
        'FN': FN,
        'FP': FP
    }




In [ ]:
# Evaluar Random Forest Básico en Test
print("EVALUACIÓN FINAL EN TEST SET")

y_pred_rf_basic = rf_basic.predict(X_test_basic)
metrics_rf_basic = clinical_metrics(y_test, y_pred_rf_basic, "Random Forest (Básico)")

In [ ]:
# Evaluar Random Forest Completo en Test
y_pred_rf_complete = rf_complete.predict(X_test_scaled)
metrics_rf_complete = clinical_metrics(y_test, y_pred_rf_complete, "Random Forest (Completo)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Paleta ──────────────────────────────────────────────────────────────
C_BASIC    = '#4C72B0'  # azul sobrio
C_COMPLETE = '#DD8047'  # naranja terra
GRAY       = '#e8e8e8'
DARK       = '#2d2d2d'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5),
                                gridspec_kw={'width_ratios': [1.4, 1]})
fig.patch.set_facecolor('white')

# ── Panel izquierdo: Cleveland dot plot ─────────────────────────────────
metrics  = ['Sensitivity', 'Specificity', 'PPV', 'NPV']
y_pos    = np.arange(len(metrics))[::-1]  # de arriba hacia abajo

b_vals = [metrics_rf_basic[m]    for m in metrics]
c_vals = [metrics_rf_complete[m] for m in metrics]

for y, bv, cv in zip(y_pos, b_vals, c_vals):
    # línea de conexión (muestra el delta)
    ax1.plot([bv, cv], [y, y], color=GRAY, lw=2.5, zorder=1)
    # puntos
    ax1.scatter(bv, y, color=C_BASIC,    s=90, zorder=3)
    ax1.scatter(cv, y, color=C_COMPLETE, s=90, zorder=3)
    # etiquetas de valor
    offset = 1.5
    ax1.text(bv - offset, y, f'{bv:.0f}', ha='right', va='center',
             fontsize=9, color=C_BASIC)
    ax1.text(cv + offset, y, f'{cv:.0f}', ha='left',  va='center',
             fontsize=9, color=C_COMPLETE)


ax1.set_yticks(y_pos)
ax1.set_yticklabels(metrics, fontsize=11)
ax1.set_xlim(0, 105)
ax1.set_xlabel('Valor (%)', fontsize=10)
ax1.set_title('Métricas clínicas', fontsize=12, fontweight='bold', pad=12)
ax1.spines[['top', 'right', 'left']].set_visible(False)
ax1.tick_params(left=False)
ax1.grid(axis='x', color=GRAY, lw=0.8)

# ── Panel derecho: lollipop horizontal mejorado ─────────────────────────
errors      = ['FN — enfermos\nno detectados', 'FP — cateterismos\ninnecesarios']
b_errs      = [metrics_rf_basic['FN'],    metrics_rf_basic['FP']]
c_errs      = [metrics_rf_complete['FN'], metrics_rf_complete['FP']]
y2          = np.array([1, 0])
OFFSET      = 0.14

for y, bv, cv in zip(y2, b_errs, c_errs):
    for val, color, sign in [(bv, C_BASIC, -1), (cv, C_COMPLETE, +1)]:
        yy = y + sign * OFFSET
        ax2.plot([0, val], [yy, yy], color=color, lw=2.5, alpha=0.8)
        ax2.scatter(val, yy, color=color, s=100, zorder=3)
        ax2.text(val + 0.3, yy, str(val), va='center', fontsize=9, color=color)

ax2.set_yticks(y2)
ax2.set_yticklabels(errors, fontsize=10)
ax2.set_xlabel('Número de casos', fontsize=10)
ax2.set_title('Errores clínicos', fontsize=12, fontweight='bold', pad=12)
ax2.spines[['top', 'right', 'left']].set_visible(False)
ax2.tick_params(left=False)
ax2.grid(axis='x', color=GRAY, lw=0.8)
ax2.set_xlim(0, max(b_errs + c_errs) * 1.3)

# ── Leyenda compartida ───────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color=C_BASIC,    label='Básico'),
    mpatches.Patch(color=C_COMPLETE, label='Completo'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.04))

plt.suptitle('Random Forest — Básico vs Completo', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

# Sección 14: Curvas ROC y análisis de threshold

## Por qué importa el threshold

Los modelos de clasificación devuelven una probabilidad de enfermedad (0 a 1).
Necesitamos un umbral de decisión (threshold) para convertir esa probabilidad en una
decisión binaria.

Hasta ahora el código usaba automáticamente `threshold = 0.5`. Cuando llamamos a:
```python
y_pred_rf_complete = rf_complete.predict(X_test_scaled)
```

...sklearn genera una probabilidad para cada paciente y aplica:
`probabilidad >= 0.5 → "enfermo"`, `probabilidad < 0.5 → "sano"`.
El 0.5 es pura convención, no tiene ningún fundamento clínico.

En medicina, mover el threshold tiene consecuencias asimétricas — los dos tipos de
error no cuestan lo mismo. Un **Falso Negativo** (FN: enviamos a casa a un paciente enfermo)
puede tener consecuencias graves o fatales. Un **Falso Positivo** (FP: cateterismo innecesario)
es costoso y molesto, pero no peligroso. Esta asimetría clínica define qué threshold elegir:

- **Threshold bajo (ej: 0.2):** alta sensibilidad — detectamos casi todos los enfermos,
  pero también enviamos al cateterismo muchos pacientes sanos (muchos falsos positivos).

- **Threshold alto (ej: 0.7):** alta especificidad — pocas alarmas innecesarias,
  pero algunos enfermos quedan sin detectar (falsos negativos), lo cual en nuestro
  contexto clínico es inaceptable.

La curva ROC muestra el trade-off entre sensibilidad y especificidad para todos los
thresholds posibles. A partir de ella definiremos tres estrategias explícitas y
compararemos su impacto clínico.

  Nota: el modelo ya lo habíamos entrenado y seleccionado usando únicamente datos de train.



  Ahora lo evaluamos por primera vez sobre el test set — los datos que el modelo nunca ha visto.

In [ ]:

y_proba_rf_basic = rf_basic.predict_proba(X_test_basic)[:, 1]
y_proba_rf_complete = rf_complete.predict_proba(X_test_scaled)[:, 1]



In [ ]:
# %%
# Nota: cuando llamamos a rf_basic.predict(X_test_basic) en la sección anterior,
# sklearn internamente aplicó probabilidad >= 0.5 → "enfermo". Ese 0.5 es pura
# convención matemática. Aquí lo hacemos explícito para poder cambiarlo.
threshold = 0.5

# Visualización: distribución de probabilidades por clase real
fig, ax = plt.subplots(figsize=(10, 6))

# Histograma separado por clase real
sns.histplot(y_proba_rf_basic[y_test == 0], bins=30, alpha=0.5, label='Sin enfermedad (real)',
             color='steelblue',kde=True, ax=ax)
sns.histplot(y_proba_rf_basic[y_test == 1], bins=30, alpha=0.5, label='Con enfermedad (real)',
             color='tomato', kde=True, ax=ax)

# Línea del threshold
ax.axvline(x=threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')

ax.set_xlabel('Probabilidad predicha de enfermedad', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title('Distribución de probabilidades predichas (modelo básico)\n'
             'Buena separación entre distribuciones = buen poder discriminativo', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

# Cómo leer este gráfico:
# - Distribución azul (sin enfermedad real): idealmente acumulada a la izquierda (probabilidades bajas)
# - Distribución roja (con enfermedad real): idealmente acumulada a la derecha (probabilidades altas)
# - Si ambas distribuciones se solapan mucho, el modelo no discrimina bien
# - La línea verde marca el threshold de decisión: pacientes a la derecha → predichos como enfermos



In [ ]:
# %%
# Calcular curvas ROC para ambos modelos
fpr_basic, tpr_basic, thresholds_basic = roc_curve(y_test, y_proba_rf_basic)
auc_basic = auc(fpr_basic, tpr_basic)

fpr_complete, tpr_complete, thresholds_complete = roc_curve(y_test, y_proba_rf_complete)
auc_complete = auc(fpr_complete, tpr_complete)

print(f"AUC Modelo Básico: {auc_basic:.3f}")
print(f"AUC Modelo Completo: {auc_complete:.3f}")




 Recordatorio:  ROC-AUC es una métrica de performance agnóstica al threshold. A diferencia de la accuracy o la sensibilidad, el AUC no depende del umbral de decisión que escojamos, sino que nos da una intuición de lo bien que discrimina el modelo en general (para todos los posibles thresholds a la vez).



  **Limitación importante**: En datasets muy desbalanceados (p.ej. 95% sanos, 5% enfermos),

  la ROC puede ser engañosamente optimista. Como el eje X usa tasa de FP relativa al total

  de sanos, un modelo que falla mucho en los enfermos puede seguir teniendo una ROC bonita.

  En esos casos, la curva Precision-Recall (Sección 14.1) es más informativa.

  En nuestro dataset (~54% enfermos, ~46% sanos) las clases están bastante equilibradas,

  así que la ROC sigue siendo una métrica válida.



  ## Cómo leer las curvas ROC — lee esto antes de ver el gráfico



  El proceso de evaluación siempre compara las predicciones del modelo con el ground truth:



  [0.1, 0.27, 0.17, 0.92, 0.02, 0.73, 0.71, 0.25] (probabilidades predichas)

  --> threshold = 0.5 -->

  [0, **0**, 0, 1, 0, 1, 1, **0**] (predicciones binarizadas)

  vs.

  [0, 1, 0, 1, 0, 1, 1, 1] (realidad)



  Los resultados dependen del threshold. La curva ROC dibuja sensibilidad vs. (1-especificidad)

  para TODOS los posibles thresholds a la vez, permitiendo comparar modelos sin fijar uno.



  Gráfico izquierdo — Curvas ROC:



  - Eje X (1 - Especificidad): proporción de pacientes SANOS que el modelo manda al cateterismo

    innecesariamente. Queremos este valor bajo (lo más a la izquierda posible).



  - Eje Y (Sensibilidad): proporción de pacientes ENFERMOS que el modelo detecta.

    Queremos este valor alto (lo más arriba posible).



  - Cada punto de la curva corresponde a un threshold diferente.



  - La diagonal punteada es el modelo aleatorio (AUC = 0.5): predice al azar.

    Cualquier modelo útil debe estar por encima de esa diagonal.



  - AUC resume todo esto en un número: 0.5 = azar, 1.0 = perfecto.

    En medicina, AUC > 0.8 se considera bueno.



  Gráfico derecho — Trade-off Sensibilidad/Especificidad con 3 estrategias de threshold:



  Aquí vemos cómo sensibilidad (verde) y especificidad (rojo) evolucionan al cambiar el threshold.

  Presentamos tres estrategias concretas:



  - Estrategia 1 — Default (0.5): convención matemática sin base clínica.



  - Estrategia 2 — Especificidad >= 90%: el threshold más bajo que garantiza que al menos

    el 90% de los sanos se clasifican correctamente. Limita cateterismos innecesarios.



  - Estrategia 3 — Sensibilidad = 100%: garantiza que ningún enfermo queda sin detectar.

    Esta es nuestra estrategia preferida porque en este problema clínico, un FN (enfermo

    no detectado) es más peligroso que un FP (cateterismo innecesario en un sano).

In [ ]:
# %%
# Visualización: Curvas ROC
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

color_basic    = 'steelblue'
color_complete = 'orange'

# ── Left: ROC curves comparison ───────────────────────────────
axes[0].plot(fpr_basic, tpr_basic, linewidth=3, color=color_basic, alpha=0.85,
             label=f'Básico (AUC = {auc_basic:.3f})')
axes[0].plot(fpr_complete, tpr_complete, linewidth=3, color=color_complete, alpha=0.85,
             label=f'Completo (AUC = {auc_complete:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.4, label='Aleatorio (AUC = 0.5)')

idx_basic_05    = np.argmin(np.abs(thresholds_basic - 0.5))
idx_complete_05 = np.argmin(np.abs(thresholds_complete - 0.5))

axes[0].scatter(fpr_basic[idx_basic_05], tpr_basic[idx_basic_05],
                s=150, c=color_basic, zorder=5, marker='o', label='Threshold=0.5 (Básico)')
axes[0].scatter(fpr_complete[idx_complete_05], tpr_complete[idx_complete_05],
                s=150, c=color_complete, zorder=5, marker='s', label='Threshold=0.5 (Completo)')

axes[0].set_xlabel('1 - Especificidad (Tasa Falsos Positivos)', fontsize=12)
axes[0].set_ylabel('Sensibilidad (Tasa Verdaderos Positivos)', fontsize=12)
axes[0].set_title('Curvas ROC: Básico vs Completo', fontsize=14)
axes[0].legend(loc='lower right', framealpha=0.95, fontsize=10)
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Right: Sensitivity / Specificity trade-off ────────────────
# Reusamos fpr_basic / tpr_basic / thresholds_basic ya calculados arriba
specificity_basic = 1 - fpr_basic

# Estrategia 1: threshold por defecto (0.5)
i_default = np.argmin(np.abs(thresholds_basic - 0.5))
th_default = thresholds_basic[i_default]

# Estrategia 2: threshold más bajo que garantiza especificidad >= 90%
_idx = np.where(specificity_basic >= 0.90)[0]
i_spec90  = _idx[-1] if len(_idx) > 0 else 0
th_spec90 = thresholds_basic[i_spec90]

# Estrategia 3: threshold más bajo con sensibilidad = 100% (cero falsos negativos)
_idx = np.where(tpr_basic >= 1.0)[0]
i_sens100  = _idx[0] if len(_idx) > 0 else 0
th_sens100 = thresholds_basic[i_sens100]

axes[1].plot(thresholds_basic, tpr_basic,         linewidth=3, color='mediumseagreen', alpha=0.85, label='Sensibilidad')
axes[1].plot(thresholds_basic, specificity_basic, linewidth=3, color='tomato',         alpha=0.85, label='Especificidad')

axes[1].axvline(th_default, color='black',     linestyle=':',  linewidth=2, alpha=0.7,
                label=f'Default 0.5  (Sen={tpr_basic[i_default]*100:.0f}%, Esp={specificity_basic[i_default]*100:.0f}%)')
axes[1].axvline(th_spec90,  color='darkorange', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Esp≥90%  th={th_spec90:.2f}  (Sen={tpr_basic[i_spec90]*100:.0f}%, Esp={specificity_basic[i_spec90]*100:.0f}%)')
axes[1].axvline(th_sens100, color='steelblue',  linestyle='--', linewidth=2, alpha=0.8,
                label=f'Sen=100%  th={th_sens100:.2f}  (Esp={specificity_basic[i_sens100]*100:.0f}%)')

axes[1].set_xlabel('Threshold de decisión', fontsize=12)
axes[1].set_ylabel('Tasa', fontsize=12)
axes[1].set_title('Trade-off Sensibilidad / Especificidad\n(Modelo Básico — 3 estrategias)', fontsize=14)
axes[1].legend(loc='center left', framealpha=0.95, fontsize=9)
axes[1].grid(alpha=0.3, linestyle='--')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ── Tabla comparativa ─────────────────────────────────────────
print("\nComparación de estrategias de threshold — Modelo Básico")
print("=" * 60)
strategies = [
    ("Default (0.5)",       th_default, i_default,  i_default),
    ("Especificidad ≥ 90%", th_spec90,  i_spec90,   i_spec90),
    ("Sensibilidad = 100%", th_sens100, i_sens100,  i_sens100),
]
print(f"{'Estrategia':<25} {'Threshold':>9} {'Sensib.':>9} {'Especif.':>9}")
print("-" * 60)
for name, th, i_tpr, i_spe in strategies:
    print(f"{name:<25} {th:>9.3f} {tpr_basic[i_tpr]*100:>8.1f}% {specificity_basic[i_spe]*100:>8.1f}%")
print("=" * 60)


---

## Sección 14.1: Curva Precision-Recall — cuando la ROC no basta

La curva ROC es útil, pero puede ser engañosamente optimista en datasets desbalanceados
(cuando una clase es mucho más rara que la otra).

La curva ROC usa en el eje X la tasa de Falsos Positivos = FP / (FP + TN). Cuando hay
muchos negativos reales (muchos pacientes sanos), el denominador es grande y la tasa de
FP parece baja aunque en términos absolutos haya muchos FP. La curva ROC no capta bien
los errores en la clase mayoritaria.

La curva Precision-Recall evita este problema porque no incluye los TN en ningún denominador:

- **Recall (eje X)** = TP / (TP + FN): ¿Qué % de enfermos detectamos?
- **Precision (eje Y)** = TP / (TP + FP): De los que predecimos enfermos, ¿qué % lo son de verdad?

Ambas métricas se centran solo en la clase positiva (los enfermos), que es exactamente
lo que nos importa clínicamente.

El baseline cambia: en la curva ROC el modelo aleatorio tiene AUC = 0.5 siempre. En la
curva Precision-Recall, el baseline es la prevalencia de la clase positiva (~45% en
nuestro dataset). Si tu Average Precision (AP) no supera ese valor claramente, el modelo
no aporta valor real.

| Situación | Recomendación |
|---|---|
| Dataset equilibrado | ROC-AUC es suficiente |
| Dataset desbalanceado (ej. enfermedad rara, 1–5%) | Usa la curva PR |
| Te importa más no perder positivos | PR más informativa |
| Necesitas comparar modelos de forma general | Ambas |

En nuestro caso el dataset está relativamente equilibrado (~45% positivos), así que ROC y
PR cuentan historias similares. Pero en problemas reales de medicina (enfermedades raras)
la diferencia es dramática: un modelo con ROC-AUC 0.85 puede tener AP de 0.3 si la clase
positiva es muy escasa. Por eso conviene ver siempre ambas.

In [ ]:
# %%
# Calcular curvas Precision-Recall para ambos modelos
precision_basic, recall_basic, _ = precision_recall_curve(y_test, y_proba_rf_basic)
ap_basic = average_precision_score(y_test, y_proba_rf_basic)

precision_complete, recall_complete, _ = precision_recall_curve(y_test, y_proba_rf_complete)
ap_complete = average_precision_score(y_test, y_proba_rf_complete)

print(f"Average Precision (AP) — Modelo Básico:   {ap_basic:.3f}")
print(f"Average Precision (AP) — Modelo Completo: {ap_complete:.3f}")
print(f"\nComparación con ROC-AUC:")
print(f"  ROC-AUC Básico:   {auc_basic:.3f}  |  AP Básico:   {ap_basic:.3f}")
print(f"  ROC-AUC Completo: {auc_complete:.3f}  |  AP Completo: {ap_complete:.3f}")

fig, ax = plt.subplots(figsize=(9, 6))

prevalencia = y_test.mean()  # Baseline de la curva PR

ax.plot(recall_basic, precision_basic, color='steelblue', linewidth=3,
        label=f'Básico (AP = {ap_basic:.3f})', alpha=0.85)
ax.plot(recall_complete, precision_complete, color='darkorange', linewidth=3,
        label=f'Completo (AP = {ap_complete:.3f})', alpha=0.85)
ax.axhline(y=prevalencia, color='tomato', linestyle='--', linewidth=2, alpha=0.7,
           label=f'Baseline aleatorio ({prevalencia:.2f})')
ax.set_xlabel('Recall (Sensibilidad)', fontsize=12)
ax.set_ylabel('Precision (PPV)', fontsize=12)
ax.set_title('Curvas Precision-Recall — Básico vs Completo', fontsize=13)
ax.legend(loc='lower left', framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()




---

## Calibración del modelo: ¿las probabilidades significan lo que creemos?

Hasta ahora hemos usado las probabilidades del modelo (output de `predict_proba`) para
tomar decisiones clínicas. Pero conviene hacerse una pregunta previa: ¿son esas
probabilidades fiables?

Cuando el modelo dice "este paciente tiene un 70% de probabilidad de enfermedad",
¿significa que de 100 pacientes similares ~70 tienen realmente la enfermedad?
Eso es lo que mide la **calibración**: la correspondencia entre las probabilidades
predichas y las frecuencias reales observadas.

### Cómo interpretar el diagrama de calibración

- **Eje X:** probabilidades predichas por el modelo (agrupadas en bins).
- **Eje Y:** fracción real de positivos en cada bin.
- **Diagonal perfecta (punteada):** calibración ideal — si predice 0.7, el 70% de esos pacientes realmente están enfermos.
- **Curva por encima de la diagonal:** el modelo subestima — cuando dice 0.7, en realidad más del 70% están enfermos.
- **Curva por debajo de la diagonal:** el modelo sobreestima — cuando dice 0.7, en realidad menos del 70% están enfermos.

> **Nota sobre Random Forests:** tienden a comprimir las probabilidades hacia valores
> intermedios (raramente predicen 0.05 o 0.95 con confianza), lo que aleja la curva
> de la diagonal. Es un comportamiento conocido y esperable.

### Por qué importa — y qué no tiene que ver con el threshold

Es importante no confundir calibración con selección de threshold. Son dos cosas distintas:

- El **threshold** se elige en función de prioridades clínicas (cuántos FN vs FP
  estamos dispuestos a asumir) y se basa en la curva ROC. No requiere calibración perfecta.
- La **calibración** importa cuando queremos usar la probabilidad predicha como un
  número clínicamente interpretable en sí mismo.

Si un médico ve en pantalla "probabilidad de enfermedad: 0.72" y toma una decisión
basándose en ese valor concreto, necesita confiar en que ese 0.72 refleja una frecuencia
real. Un modelo mal calibrado puede emitir 0.72 cuando la frecuencia real es 0.45 —
la decisión clínica estaría basada en un número incorrecto.

La calibración es, por tanto, un **requisito de seguridad clínica** cuando las
probabilidades se comunican directamente: no un detalle técnico, y no un prerequisito
para fijar un threshold.

In [ ]:
# %%
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_proba, model_name, color in zip(
    axes,
    [y_proba_rf_basic, y_proba_rf_complete],
    ['Modelo Básico (sin gammagrafía)', 'Modelo Completo (con gammagrafía)'],
    ['steelblue', 'darkorange']):

    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=8, strategy='quantile')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='Calibración perfecta')
    ax.plot(prob_pred, prob_true, 'o-', color=color, linewidth=2.5, markersize=9,
            label=model_name, )

    ax.set_xlabel('Probabilidad predicha por el modelo', fontsize=11)
    ax.set_ylabel('Fracción de positivos reales', fontsize=11)
    ax.set_title(f'Diagrama de Calibración\n{model_name}', fontsize=12)
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3, linestyle='--')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\nRecuerda: los Random Forests tienden a comprimir probabilidades hacia valores")
print("intermedios (alejándose de 0 y 1). Si la calibración no es perfecta, el threshold")
print("seleccionado puede no corresponder exactamente con la frecuencia real de enfermedad.")
print("Existen técnicas para corregirlo (Platt Scaling, Isotonic Regression), aunque")
print("quedan fuera del alcance de este script.")




In [ ]:
# %%
# Análisis de diferentes thresholds con el modelo básico
#
# Nota: roc_curve() ya devuelve internamente todos los thresholds y sus métricas.
# Hacemos este bucle manualmente para que veáis exactamente qué hay "dentro" de esa función
# y cómo se construyen las métricas punto a punto. En producción usaríais roc_curve() directamente.

thresholds_to_test = np.linspace(0, 1, num=51)
results = []

for threshold in thresholds_to_test:
    y_pred_threshold = (y_proba_rf_basic >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred_threshold)
    TN, FP, FN, TP = cm.ravel()

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0

    results.append({
        'Threshold': threshold,
        'Sensitivity': sensitivity * 100,
        'Specificity': specificity * 100,
        'Accuracy': (TP + TN) / (TP + TN + FP + FN) * 100,
        'PPV': ppv * 100,
        'NPV': npv * 100,
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'Total_Errors': FN + FP
    })

results_df = pd.DataFrame(results)
print("Análisis de diferentes thresholds (Modelo Básico):")
print("=" * 80)
print("Cómo leer esta tabla:")
print("  - Cada fila = un threshold distinto (de 0.0 a 1.0 en pasos de 0.02)")
print("  - Sensitivity sube al bajar el threshold (clasificamos más como enfermos)")
print("  - Specificity sube al subir el threshold (clasificamos más como sanos)")
print("  - Busca la fila donde Sensitivity = 100.0 con el threshold más alto:")
print("    ese es nuestro umbral óptimo para no perder ningún enfermo.")
print("  - FN = enfermos no detectados (queremos este = 0)")
print("  - FP = sanos enviados al cateterismo innecesariamente (queremos este bajo)")
print("=" * 80)
print(results_df.round(3).to_string(index=False))




---

## Interpretación clínica de los thresholds

La tabla de arriba muestra cómo evolucionan sensibilidad, especificidad, PPV y NPV
para cada valor de threshold, permitiéndonos identificar el punto que cumple nuestros
requisitos clínicos.

En nuestro caso la prioridad es clara: no podemos permitirnos perder ningún enfermo.
Un **falso negativo** implica mandar a casa a un paciente que necesita cateterismo —
consecuencias potencialmente graves. Un **falso positivo** implica un cateterismo
innecesario: molesto y costoso, pero no peligroso.

La estrategia elegida es **sensibilidad = 100%**, correspondiente al threshold más bajo
que garantiza cero falsos negativos. Con ese umbral, el modelo descarta aproximadamente
un 60–65% de los cateterismos innecesarios en pacientes sanos, manteniendo detectados
el 100% de los enfermos.

> Esto tendría que validarse en un dataset externo antes de llevarlo a producción real, así como calcular el threshold tal que sensitivity == 100% en validación cruzada.

---
## Sección 15: Importancia de las variables

El Random Forest puede decirnos qué variables son más útiles para predecir la enfermedad.
La métrica que usa se llama **Mean Decrease in Impurity (MDI)**: mide cuánto reduce
cada variable la impureza de los nodos del árbol al ser usada para hacer splits,
promediada sobre todos los árboles del bosque.

- Una variable con MDI **alto** aparece frecuentemente en los splits superiores de los
  árboles y separa bien las clases (enfermos vs. sanos).
- Una variable con MDI **bajo** apenas contribuye: el modelo funciona casi igual sin ella.

Los valores están normalizados para sumar 1. No es una medida de causalidad —
una variable "importante" para el modelo no necesariamente causa la enfermedad;
puede estar correlacionada con otra variable que sí la causa.

In [ ]:
# %%
# Feature Importance — ambos modelos en una sola figura
importances_basic    = rf_basic.feature_importances_
indices_basic        = np.argsort(importances_basic)[::-1]

importances_complete = rf_complete.feature_importances_
indices_complete     = np.argsort(importances_complete)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(len(importances_basic)), importances_basic[indices_basic],
             color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(importances_basic)))
axes[0].set_yticklabels([basic_features[i] for i in indices_basic])
axes[0].invert_yaxis()
axes[0].set_xlabel('Importancia', fontsize=12)
axes[0].set_title('Random Forest Básico', fontsize=13)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].barh(range(len(importances_complete)), importances_complete[indices_complete],
             color='darkorange', alpha=0.8)
axes[1].set_yticks(range(len(importances_complete)))
axes[1].set_yticklabels([complete_features[i] for i in indices_complete])
axes[1].invert_yaxis()
axes[1].set_xlabel('Importancia', fontsize=12)
axes[1].set_title('Random Forest Completo', fontsize=13)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

fig.suptitle('Importancia de variables', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()



  ### Interpretación Clínica del Feature Importance (Modelo Completo):



  Las tres variables con mayor importancia son:



  - thal (gammagrafía cardíaca): Aparece como la variable más determinante, lo que sugiere su relevancia diagnóstica en la detección de enfermedad coronaria y apoya su indicación en casos ambiguos tras prueba de esfuerzo no concluyente.



  - thalach (frecuencia cardíaca máxima alcanzada): Indica la capacidad funcional del corazón durante el esfuerzo. Una frecuencia máxima reducida podría estar relacionada con limitación coronaria.



  - oldpeak (depresión del segmento ST): Esta medida electrocardiográfica durante el esfuerzo tiende a reflejar isquemia miocárdica, donde valores elevados suelen asociarse con obstrucción coronaria significativa.



  Otras variables como el tipo de dolor torácico (cp_4), edad y colesterol también muestran contribuciones notables. En contraste, variables como sexo y resultados de ECG en reposo presentan menor peso predictivo, aunque contribuyen al modelo global.



  Este patrón de importancia parece coherente con el conocimiento clínico: las pruebas funcionales (esfuerzo, gammagrafía) y los signos de isquemia (como la depresión del ST o defectos de perfusión) tienden a ser más predictivos que los factores de riesgo basales (edad, sexo, colesterol) para identificar enfermedad coronaria establecida que requiere intervención.

---

## Sección 16: Threshold óptimo por CV y entrenamiento final

Ahora que tenemos el mejor modelo, el proceso correcto es:

1. Seleccionar el threshold usando **solo datos de train** (cross-validation)
2. Estimar con CV qué sensibilidad/especificidad podemos esperar en producción
3. Evaluar el impacto clínico en el test set (una sola vez)
4. Entrenar el modelo final con todos los datos (train + test)

El threshold **no puede seleccionarse usando el test set**. Si lo hiciésemos, estaríamos
ajustando una decisión de diseño a esos datos concretos y el threshold no generalizaría
a nuevos pacientes — el mismo principio que aplica a no usar el test set para entrenar
el modelo.

In [ ]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# PASO 1: Selección del threshold y estimación de rendimiento por CV
# ─────────────────────────────────────────────────────────────────────────────
#
# Por cada fold:
#   - Entrenamos el modelo en el sub-train del fold
#   - Buscamos el threshold que maximiza sensibilidad en la validación (sens=100%)
#   - Guardamos tanto el threshold como (y_val, y_proba) para el paso 2
#
# La estrategia elegida es sensibilidad = 100%: queremos el threshold más alto que
# todavía captura a todos los enfermos. Ese umbral varía fold a fold según los datos.
# Usamos la MEDIANA de los thresholds de los 5 folds porque es más robusta que la
# media ante valores extremos. Otra opción sería tomar el threshold mínimo de los folds
# (el más conservador de todos), lo que garantizaría sensibilidad=100% en TODOS los
# folds por separado. La desventaja es que puede ser excesivamente bajo y generar más
# falsos positivos. La mediana es un compromiso razonable entre robustez y especificidad.

# StratifiedKFold — divide los datos en 5 grupos (folds) de forma estratificada,
# es decir, manteniendo la proporción enfermos/sanos en cada fold.
# Esto es importante en datasets con clases desbalanceadas: si no estratificamos,
# algún fold podría tener muy pocos enfermos y los resultados serían inestables.
# shuffle=True mezcla los datos antes de dividirlos para evitar patrones por orden de filas.
cv_threshold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
thresholds_por_fold = []
folds_data = []  # Guardamos (y_val, y_proba) para estimar rendimiento con el threshold final

print("Buscando threshold óptimo mediante Cross-Validation (solo datos de TRAIN)...")
print("="*70)

for fold, (train_idx, val_idx) in enumerate(cv_threshold.split(X_train_basic, y_train)):
    X_tr_fold  = X_train_basic.iloc[train_idx]
    X_val_fold = X_train_basic.iloc[val_idx]
    y_tr_fold  = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]

    modelo_fold = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    modelo_fold.fit(X_tr_fold, y_tr_fold)

    y_proba_fold = modelo_fold.predict_proba(X_val_fold)[:, 1]
    fpr_f, tpr_f, thr_f = roc_curve(y_val_fold, y_proba_fold)

    # Threshold más alto que mantiene sensibilidad = 100%
    idx_100 = np.where(tpr_f >= 1.0)[0]
    th_fold = thr_f[idx_100[0]] if len(idx_100) > 0 else thr_f[np.argmax(tpr_f)]
    thresholds_por_fold.append(th_fold)
    folds_data.append((y_val_fold, y_proba_fold))

    sens_fold = tpr_f[idx_100[0]] if len(idx_100) > 0 else tpr_f[np.argmax(tpr_f)]
    print(f"  Fold {fold+1}: threshold = {th_fold:.3f}  → Sensibilidad en validación = {sens_fold*100:.0f}%")

# Usamos la mediana (más robusta que la media ante valores atípicos entre folds)
optimal_threshold_basic = float(np.median(thresholds_por_fold))
print(f"\nThreshold final (mediana de 5 folds): {optimal_threshold_basic:.3f}")
print(f"Rango de thresholds por fold: [{min(thresholds_por_fold):.3f} – {max(thresholds_por_fold):.3f}]")
print("Este threshold se estimó ÚNICAMENTE con datos de train — el test set sigue intacto.")
print()


# ─────────────────────────────────────────────────────────────────────────────
# PASO 2: Estimación del rendimiento esperado en producción (con el threshold final)
# ─────────────────────────────────────────────────────────────────────────────
# Aplicamos el threshold mediano a los datos de validación de cada fold
# para estimar qué sensibilidad/especificidad podemos esperar en producción.
# Esta es la estimación más honesta disponible (sin tocar el test set).

sensitividades_cv = []
especificidades_cv = []

for y_val_fold, y_proba_fold in folds_data:
    y_pred_cv = (y_proba_fold >= optimal_threshold_basic).astype(int)
    cm_cv = confusion_matrix(y_val_fold, y_pred_cv)
    TN_cv, FP_cv, FN_cv, TP_cv = cm_cv.ravel()
    sensitividades_cv.append(TP_cv / (TP_cv + FN_cv) if (TP_cv + FN_cv) > 0 else 0)
    especificidades_cv.append(TN_cv / (TN_cv + FP_cv) if (TN_cv + FP_cv) > 0 else 0)

print(f"\nRendimiento esperado en producción (CV con threshold={optimal_threshold_basic:.3f}):")
print(f"  Sensibilidad:  {np.mean(sensitividades_cv)*100:.1f}% ± {np.std(sensitividades_cv)*100:.1f}%")
print(f"  Especificidad: {np.mean(especificidades_cv)*100:.1f}% ± {np.std(especificidades_cv)*100:.1f}%")
print("\nEsta es la mejor")
print("aproximación disponible antes de ver datos reales de producción.")

# PASO 3: Evaluar impacto clínico con el threshold seleccionado (en test)
# ───────────────────────────────────────────────────────────────────────
y_pred_rf_basic_optimal = (y_proba_rf_basic >= optimal_threshold_basic).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_rf_basic_optimal)
TN_opt, FP_opt, FN_opt, TP_opt = cm_opt.ravel()

total_sanos = TN_opt + FP_opt
porcentaje_evitados = (TN_opt / total_sanos) * 100 if total_sanos > 0 else 0

print("\nIMPACTO CLÍNICO — MODELO BÁSICO CON THRESHOLD ÓPTIMO")
print("="*70)
print(f"Threshold aplicado: {optimal_threshold_basic:.3f}")
print(f"\nPacientes sin enfermedad (n={total_sanos}):")
print(f"  Cateterismos evitados:      {TN_opt} ({porcentaje_evitados:.1f}%)")
print(f"  Cateterismos innecesarios:  {FP_opt} ({(FP_opt/total_sanos*100) if total_sanos>0 else 0:.1f}%)")
print(f"\nPacientes con enfermedad (n={TP_opt + FN_opt}):")
print(f"  Correctamente detectados:   {TP_opt} ({(TP_opt/(TP_opt+FN_opt)*100) if (TP_opt+FN_opt)>0 else 0:.1f}%)")
print(f"  Casos no detectados (FN):   {FN_opt} ({(FN_opt/(TP_opt+FN_opt)*100) if (TP_opt+FN_opt)>0 else 0:.1f}%)")
print(f"\nBeneficio estimado:")
print(f"  Reducción de cateterismos en sanos: {porcentaje_evitados:.1f}%")
print(f"  Ahorro económico estimado: {TN_opt * 3000:,} EUR")
print(f"  Reducción de complicaciones: ~{int(TN_opt * 0.02)} pacientes")
print("="*70)

# PASO 4: Entrenar modelo final con TODOS los datos (train + test)
# ────────────────────────────────────────────────────────────────
# Ahora que el threshold ya está fijado (no usamos test para nada más),
# reentrenamos con todos los datos disponibles para maximizar la información.
X_all_basic = pd.concat([X_train_basic, X_test_basic], axis=0)
y_all       = pd.concat([y_train, y_test], axis=0)

rf_basic_final = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_basic_final.fit(X_all_basic, y_all)

os.makedirs('./models', exist_ok=True)
joblib.dump(rf_basic_final, './models/rf_heart_basic_final.pkl')
joblib.dump(optimal_threshold_basic, './models/optimal_threshold_basic.pkl')

print(f"\nModelo final guardado. Threshold óptimo: {optimal_threshold_basic:.3f}")
print("Listo para producción (o validación externa).")




   ---



   # FIN DEL SCRIPT



   ## Conceptos Cubiertos:



   - Preparación de datos completa (missing values, encoding, scaling)



   - Identificación y manejo de data leakage (variable `ca`, pacientes repetidos)



   - CRÍTICO: Imputación y escalado DESPUÉS del split para evitar data leakage



   - Dos estrategias de modelado (Básico vs Completo)



   - Train/Test Split y Cross-Validation



   - Modelos baseline y 4 algoritmos de clasificación



   - Evaluación correcta (Train/Test/CV)



   - Métricas clínicas (Sensitivity, Specificity, PPV, NPV)



   - Curvas ROC y análisis de threshold



   - Impacto clínico real (ahorro económico, reducción de complicaciones)



   - Trade-offs clínicos (casos detectados vs procedimientos evitados)



   - Entrenar el modelo final y guardar para validación externa / Producción



  ---



  ## ¿Qué explorar a continuación?



  Este script cubre el pipeline fundamental de ML supervisado para clasificación. Los siguientes pasos naturales son:



  - Gradient Boosting (XGBoost, LightGBM): alternativas a Random Forest que en muchos problemas tabulares dan mejor accuracy con más eficiencia. Mismo paradigma de ensemble pero con árboles que se construyen secuencialmente (corrigiendo los errores del anterior).



  - Ajuste de hiperparámetros (GridSearchCV, RandomizedSearchCV): buscar sistemáticamente los valores óptimos de `n_estimators`, `max_depth`, etc. en lugar de usar valores por defecto.



  - SHAP (SHapley Additive exPlanations): explicar por qué el modelo toma cada decisión concreta para cada paciente — crucial en medicina para que el clínico entienda y confíe en la predicción.



  - Manejo de desbalanceo (SMOTE, class_weight): cuando la clase positiva es muy minoritaria (ej. enfermedad rara), técnicas específicas para que el modelo no ignore esa clase.



  - Nested cross-validation: evaluación estadísticamente imparcial cuando no se tiene suficiente datos para separar un test set independiente.



  - Calibración de probabilidades (Platt Scaling, Isotonic Regression): corregir la tendencia de Random Forest a comprimir probabilidades, para que el threshold tenga un significado clínico más directo.

  - Y una infinidad de tecnicas que salen nuevas cada dia...



PROFESOR: MARC PALOMER CADENAS